# The planning problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/05_problem.ipynb)

`PlanningProblem` is the declarative task specification read by every planner, controller evaluator, and learner—whether deterministic or stochastic, finite- or infinite-horizon, continuous-time or on a clock (`StepPlanningProblem`).

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `PlanningProblem`: fields, properties, and public methods

A `PlanningProblem` states *what* to achieve and under *what* constraints, never how to solve it:

$$\min J = \int_{t_0}^{t_0 + t_f} e^{-\rho (t - t_0)} g(x, u, t)\, dt + e^{-\rho t_f} h(x(t_0 + t_f))
\quad \text{s.t.} \quad \dot x = f(x, u, t),\; x(t) \in X,\; u(t) \in U,\;
x(t_0) \in X_0,\; x(t_0 + t_f) \in X_f.$$

### Constructor arguments and attributes

| Field / Property | Type | Default | Meaning |
| --- | --- | --- | --- |
| `sys` | `System` | required | Plant model |
| `x0` | `array (n,) \| Distribution \| None` | from `X0`, else `sys.x0` | Initial state point (or mean when a `Distribution` is given) |
| `start_law` | `Distribution \| None` | `None` | Initial-state distribution when `x0` was given as a `Distribution` |
| `X0` | `Set` | `SingletonSet(x0)` or `start_law.support` | Hard initial state constraint $x(t_0) \in X_0$ |
| `X`, `U` | `Set`, `InputSet \| BoxSet` (a box is wrapped) | unconstrained | Hard state and control constraints along the horizon |
| `Xf` | `Set \| array (n,) \| None` | `None` | Terminal constraint $x(t_0 + t_f) \in X_f$ (an array is wrapped in `SingletonSet`) |
| `cost` | `CostFunction \| None` | `None` | Running and terminal cost (`None` for pure feasibility) |
| `horizon` | `float \| None` | `None` | Horizon length in seconds (`None` or `np.inf` for infinite horizon) |
| `t0`, `discount_rate` | `float`, `float` | `0.0`, `0.0` | Start time $t_0$ and continuous discount rate $\rho \ge 0$ |
| `disturbances` | `dict[str, System]` | `{}` | Source blocks wired onto named plant input ports |
| `distributions` (constructor and attribute) | `Mapping \| None` | `{}` | Parameter `Distribution`s in the shape of `sys.params` / `cost.params` |
| `criterion` | `"expectation" \| "worst_case" \| ("cvar", alpha)` | `"expectation"` | Scalar reduction of $J$ across Monte Carlo trials |
| `penalize_failure`, `failure_cost` | `bool`, `float \| None` | `False`, `None` | Whether to add a failure penalty to $J$ when a run leaves `X`/`U` or misses `Xf` |
| `problem.params` | `dict` | live view | Full parameter tree of numbers (`plant`, `cost`, `X`, `U`, `Xf`, `x0`, `t0`, `discount_rate`) |
| `problem.distributions` | `dict` | the laws | The laws over uncertain numbers, in the tree's own shape |
| `problem.is_stochastic` | `bool` | derived | `True` if `start_law`, `distributions`, or random `disturbances` are present |

### Public methods on `PlanningProblem`

| Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `problem.horizon_kind()` | — | `"finite" \| "infinite"` | `"finite"` when `horizon` is finite, `"infinite"` when it is `None` or `inf` |
| `problem.goal_point()` | — | `ndarray (n,)` | `Xf`'s point, else the cost's `x_bar` when inside `X`, else the start |
| `problem.discount_factor(dt)` | `dt: float` | `float` | Per-tick discount $\gamma = e^{-\rho\,\Delta t} \in (0, 1]$ |
| `problem.cost_bound(...)` | `horizon=None, *, discount_rate=None, dt=None, params=None` | `float` | Analytical upper bound on finite-run cost over $X \times U$ |
| `problem.failure_cost_at(...)` | `horizon=None, *, discount_rate=None, dt=None, params=None` | `float` | Explicit `failure_cost` if set, else `cost_bound(...)` |
| `problem.replace(**changes)` | keyword overrides of constructor fields | `PlanningProblem` | New `PlanningProblem` with updated fields, sharing unchanged objects |
| `problem.drawn_paths()` | — | `list[tuple[str, ...]]` | Tuple key paths of all stochastic leaves in `problem.params` |
| `problem.sample_x0(key, n=None)` | `key`, `n: int \| None = None` | `array (n_dim,)` or `(n, n_dim)` | Draw initial state(s) from `start_law`, else `x0` |
| `problem.realize(key=None, n=1)` | `key=None`, `n: int = 1` | `dict` | Draw a concrete parameter tree, `n` batched draws (the means when `key=None`) |
| `problem.nominal()` | — | `PlanningProblem` | Deterministic problem with every law replaced by its mean and random sources disabled |
| `problem.discretize(dt, ...)` | `dt: float, *, solver="rk4", substeps=1` | `StepPlanningProblem` | Explicit zero-order-hold discretization at period `dt` |
| `problem.compile(controller, ...)` | `controller: System`, `dt=None, *, solver="rk4", substeps=None, backend="auto", episode_length=None` | `ProblemEvaluator` | Compile the closed-loop evaluator of the controller |

In [ ]:
import numpy as np

from minilink import BallSet, Pendulum, PlanningProblem, QuadraticCost

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(
    plant, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1), x_bar=UP
)

problem = PlanningProblem(
    plant,
    x0=np.array([0.0, 0.0]),
    cost=cost,
    X=plant.state.box,
    U=plant.input_box,
    Xf=BallSet(UP, 0.2),
    horizon=5.0,
)
problem

In [ ]:
problem.X0, problem.goal_point(), problem.horizon_kind()

In [ ]:
problem.cost_bound(), problem.failure_cost_at()

`problem.replace(**changes)` derives a modified problem without mutating the original—for example, turning the finite-horizon swing-up into an infinite-horizon discounted regulation problem for value iteration or LQR:

In [ ]:
infinite_problem = problem.replace(horizon=None, Xf=None, discount_rate=0.5)
infinite_problem.horizon_kind(), infinite_problem.discount_factor(0.05), infinite_problem.cost_bound()

## 2. Stochastic problems: `is_stochastic`, `drawn_paths`, `sample_x0`, `realize`, and `nominal`

A `PlanningProblem` becomes stochastic when any of three sources of uncertainty is provided:
1. `x0=` a `Distribution` (initial state distribution; `X0` defaults to its support);
2. `distributions=` a nested dict of `Distribution`s over plant or cost parameters (domain randomization);
3. `disturbances=` a dict mapping plant input port names to random source blocks (such as `WhiteNoise`).

In [ ]:
from minilink import Uniform

random_problem = PlanningProblem(
    plant,
    x0=Uniform([-0.2, -0.2], [0.2, 0.2]),
    cost=cost,
    horizon=5.0,
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
random_problem.is_stochastic, random_problem.drawn_paths(), random_problem.X0

In [ ]:
# Draw 1 start (shape (2,)) vs 3 starts (shape (3, 2))
random_problem.sample_x0(0), random_problem.sample_x0(0, n=3)

In [ ]:
# Nominal deterministic problem (all laws at their mean)
nom = random_problem.nominal()
nom.is_stochastic, nom.x0, nom.sys.params["m"]

Adding a `WhiteNoise` block on the disturbance port `"d"` of `NoisyPendulum`:

In [ ]:
from minilink import NoisyPendulum, WhiteNoise

noisy_plant = NoisyPendulum()
noisy_cost = QuadraticCost.from_system(
    noisy_plant, Q=np.eye(2), R=np.diag([1.0, 0.0, 0.0])
)

noisy_problem = PlanningProblem(
    noisy_plant,
    x0=np.zeros(2),
    cost=noisy_cost,
    horizon=5.0,
    disturbances={"d": WhiteNoise(1, psd=0.04, sample_period=0.01, seed=1)},
)
(
    list(noisy_plant.inputs),
    noisy_problem.drawn_paths(),
    noisy_problem.realize(0, 3)["disturbance"],
)

## 3. `StepPlanningProblem`: the problem on a clock

Solvers that work tick by tick (dynamic programming, reinforcement learning, discrete rollouts) read the problem on a clock:
- **`problem.discretize(dt, solver="rk4", substeps=1) -> StepPlanningProblem`**: the textbook's discrete-time optimal control problem, $x_{k+1} = F(x_k, u_k)$, $J = \sum_{k<N} \gamma^k c_k + \gamma^N h(x_N)$, with $N = t_f / \Delta t$ ticks, $\gamma = e^{-\rho \Delta t}$, a `SampledSystem` plant, and a `SampledCost`.
- **The step problem answers its own equations** (`problem.compile(controller)` is the controller's evaluator, the next tutorial):

| `StepPlanningProblem` Attribute / Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `step_problem.dt`, `step_problem.n_steps`, `step_problem.gamma` | — | `float \| None`, `int \| None`, `float` | Period $\Delta t$ (`None` for a native problem's ticks), horizon in ticks $N$, discount per tick $\gamma$ |
| `step_problem.sys`, `step_problem.cost` | — | `SampledSystem`, `SampledCost` | The plant on the clock and the stage cost $c_k$ |
| `step_problem.X`, `step_problem.U`, `step_problem.Xf` | — | `Set`, `InputSet`, `Set \| None` | State, input, and terminal sets |
| `step_problem.params` | — | `dict` | Parameter tree (`plant`, `cost`, the sets', `x0`, `t0`, `gamma`) |
| `step_problem.transition(x, u, k, params=None)` | `x: (n,)`, `u: (m,)`, `k: int`, `params=None` | `(x_next: array (n,), c_k: float)` | Tick $k$: the next state $x_{k+1} = F(x_k, u_k)$ and the stage cost $c_k$ |
| `step_problem.terminal(x, t=None, params=None)` | `x: (n,)`, `t: float \| None = None`, `params=None` | `float` | Terminal cost $h(x_N)$ (`0.0` on an infinite horizon) |
| `step_problem.rollout(controller, ...)` | `controller: System \| Callable[[x, t_k], u]`, `x0=None`, `params=None`, `n_steps=None` | `Trajectory`: `x` `(n, N+1)`, `u` `(m, N+1)`, `signals["J"]` `(1, N+1)` | Roll out the controller over $N$ ticks; the running cost $J_k = \sum_{j<k} \gamma^j c_j$ ends at the total discounted cost $J$ |
| `step_problem.cost_bound(n_steps=None, *, gamma=None, params=None)` | `n_steps`, `gamma`, `params` | `float` | Discrete-time cost bound $\sum_{k<N} \gamma^k c_{max} + \gamma^N h_{max}$ |
| `step_problem.failure_cost_at(n_steps=None, *, gamma=None, params=None)` | `n_steps`, `gamma`, `params` | `float` | Explicit `failure_cost` if set, else `cost_bound(...)` |

In [ ]:
step_problem = problem.discretize(0.05)
(
    type(step_problem.sys).__name__,
    type(step_problem.cost).__name__,
    step_problem.n_steps,
    step_problem.gamma,
    step_problem.cost_bound(),
)

In [ ]:
x = np.zeros(2)
u = np.array([1.0])
x_next, c_0 = step_problem.transition(x, u, k=0)
x_next, c_0, step_problem.terminal(UP), step_problem.cost_bound()

In [ ]:
# The same tick from the discretized problem built again: one problem, one transition
problem.discretize(0.05).transition(x, u, k=0)

In [ ]:
traj = step_problem.rollout(lambda x, t_k: np.array([2.0]))
traj.x.shape, traj.u.shape, traj.signals["J"][0, -1]

Next, [`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb): `ProblemEvaluator`, `Evaluation`, `compare`, and `sys.compile()`.